In [7]:
!pip install transformers torch scikit-learn -q

from google.colab import drive
drive.mount('/content/drive')

import pandas as pd
import numpy as np
import torch
import os
import re
import json
import gc
import warnings
warnings.filterwarnings('ignore')

from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    accuracy_score,
    precision_recall_fscore_support,
    classification_report
)
from transformers import (
    BertTokenizer, BertForSequenceClassification,
    RobertaTokenizer, RobertaForSequenceClassification,
    Trainer, TrainingArguments
)
from torch.utils.data import Dataset

# ── Check GPU ──────────────────────────────────────
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Device: {device}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
else:
    print("WARNING: No GPU — go to Runtime → Change runtime type → GPU")

# ── Paths ──────────────────────────────────────────
LABELLED_PATH = "/content/drive/MyDrive/Original Reddit Data/Labelled Data"
SAVE_PATH     = "/content/drive/MyDrive/"

# ── Label mapping ──────────────────────────────────
label2id = {
    'Drug and Alcohol' : 0,
    'Early Life'       : 1,
    'Personality'      : 2,
    'Trauma and Stress': 3
}
id2label = {v: k for k, v in label2id.items()}
print(f"Labels: {label2id}")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Device: cuda
GPU: Tesla T4
Labels: {'Drug and Alcohol': 0, 'Early Life': 1, 'Personality': 2, 'Trauma and Stress': 3}


In [8]:
# ── Load Part B and create train/test split ────────
def clean_for_bert(text):
    if not isinstance(text, str): return ''
    text = re.sub(r'http\S+|www\S+', '', text)
    text = re.sub(r'<.*?>', '', text)
    text = re.sub(r'[^a-zA-Z\s]', '', text)
    return text.lower().strip()

label_files = {
    "Drug and Alcohol" : LABELLED_PATH + "/LD DA 1.csv",
    "Early Life"       : LABELLED_PATH + "/LD EL1.csv",
    "Personality"      : LABELLED_PATH + "/LD PF1.csv",
    "Trauma and Stress": LABELLED_PATH + "/LD TS 1.csv"
}

dfs_b = []
for label, filepath in label_files.items():
    try:
        temp = pd.read_csv(filepath)
        temp['Label'] = label
        dfs_b.append(temp)
        print(f"✓ {label}: {temp.shape[0]} rows")
    except Exception as e:
        print(f"✗ {label}: {e}")

df_b = pd.concat(dfs_b, ignore_index=True)
df_b = df_b.dropna(subset=['selftext'])
if 'CAT 1' in df_b.columns:
    df_b = df_b.drop(columns=['CAT 1'])

df_b['full_text'] = (
    df_b['title'].fillna('') + ' ' +
    df_b['selftext'].fillna('')
)
df_b['bert_text'] = df_b['full_text'].apply(clean_for_bert)
df_b['label_id']  = df_b['Label'].map(label2id)

# Stratified 80/20 split
train_df, test_df = train_test_split(
    df_b, test_size=0.2,
    random_state=42,
    stratify=df_b['label_id']
)

print(f"\nTrain: {len(train_df)} posts")
print(f"Test:  {len(test_df)} posts")
print(f"\nTrain labels: {train_df['Label'].value_counts().to_dict()}")
print(f"Test labels:  {test_df['Label'].value_counts().to_dict()}")

✓ Drug and Alcohol: 223 rows
✓ Early Life: 200 rows
✓ Personality: 200 rows
✓ Trauma and Stress: 200 rows

Train: 640 posts
Test:  160 posts

Train labels: {'Drug and Alcohol': 160, 'Early Life': 160, 'Personality': 160, 'Trauma and Stress': 160}
Test labels:  {'Personality': 40, 'Early Life': 40, 'Trauma and Stress': 40, 'Drug and Alcohol': 40}


In [9]:
# ── Dataset class and metrics ──────────────────────
class MentalHealthDataset(Dataset):
    def __init__(self, texts, labels, tokenizer, max_length=256):
        self.encodings = tokenizer(
            list(texts), truncation=True,
            padding=True, max_length=max_length,
            return_tensors='pt'
        )
        self.labels = torch.tensor(list(labels), dtype=torch.long)

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx):
        item = {k: v[idx] for k, v in self.encodings.items()}
        item['labels'] = self.labels[idx]
        return item

def compute_metrics(pred):
    labels = pred.label_ids
    preds  = pred.predictions.argmax(-1)
    precision, recall, f1, _ = precision_recall_fscore_support(
        labels, preds, average='macro'
    )
    acc = accuracy_score(labels, preds)
    return {
        'accuracy' : round(acc, 4),
        'f1'       : round(f1, 4),
        'precision': round(precision, 4),
        'recall'   : round(recall, 4)
    }

def get_training_args(output_dir):
    return TrainingArguments(
        output_dir                  = output_dir,
        num_train_epochs            = 3,
        per_device_train_batch_size = 16,
        per_device_eval_batch_size  = 16,
        learning_rate               = 2e-5,
        weight_decay                = 0.01,
        eval_strategy               = 'epoch',
        save_strategy               = 'epoch',
        load_best_model_at_end      = True,
        logging_steps               = 10,
        report_to                   = 'none'
    )

print("Dataset class, metrics and training args ready")

Dataset class, metrics and training args ready


In [10]:
# ── BERT fine-tuning and evaluation ───────────────
print("Loading BERT...")
bert_tokenizer = BertTokenizer.from_pretrained('bert-base-uncased')
bert_model     = BertForSequenceClassification.from_pretrained(
    'bert-base-uncased', num_labels=4,
    id2label=id2label, label2id=label2id,
    ignore_mismatched_sizes=True
)

bert_train = MentalHealthDataset(
    train_df['bert_text'], train_df['label_id'], bert_tokenizer
)
bert_test  = MentalHealthDataset(
    test_df['bert_text'], test_df['label_id'], bert_tokenizer
)

bert_trainer = Trainer(
    model           = bert_model,
    args            = get_training_args('/content/bert_results'),
    train_dataset   = bert_train,
    eval_dataset    = bert_test,
    compute_metrics = compute_metrics
)

print("Training BERT — ~20 minutes on GPU\n")
bert_trainer.train()

# Evaluate
bert_preds       = bert_trainer.predict(bert_test)
bert_pred_labels = bert_preds.predictions.argmax(-1)
bert_true_labels = bert_preds.label_ids

print("\nBERT Classification Report:")
print(classification_report(
    bert_true_labels, bert_pred_labels,
    target_names=list(label2id.keys())
))

bert_results = {
    'accuracy' : round(accuracy_score(bert_true_labels, bert_pred_labels), 4),
    'f1_macro' : round(classification_report(bert_true_labels, bert_pred_labels, output_dict=True)['macro avg']['f1-score'], 4),
    'precision': round(classification_report(bert_true_labels, bert_pred_labels, output_dict=True)['macro avg']['precision'], 4),
    'recall'   : round(classification_report(bert_true_labels, bert_pred_labels, output_dict=True)['macro avg']['recall'], 4)
}
print(f"\nBERT → Accuracy: {bert_results['accuracy']} | F1: {bert_results['f1_macro']}")

# Free GPU memory
del bert_model, bert_trainer
gc.collect()
torch.cuda.empty_cache()

Loading BERT...


tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

...

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

There were missing keys in the checkpoint model loaded: ['bert.embeddings.LayerNorm.weight', 'bert.embeddings.LayerNorm.bias', 'bert.encoder.layer.0.attention.output.LayerNorm.weight', 'bert.encoder.layer.0.attention.output.LayerNorm.bias', 'bert.encoder.layer.0.output.LayerNorm.weight', 'bert.encoder.layer.0.output.LayerNorm.bias', 'bert.encoder.layer.1.attention.output.LayerNorm.weight', 'bert.encoder.layer.1.attention.output.LayerNorm.bias', 'bert.encoder.layer.1.output.LayerNorm.weight', 'bert.encoder.layer.1.output.LayerNorm.bias', 'bert.encoder.layer.2.attention.output.LayerNorm.weight', 'bert.encoder.layer.2.attention.output.LayerNorm.bias', 'bert.encoder.layer.2.output.LayerNorm.weight', 'bert.encoder.layer.2.output.LayerNorm.bias', 'bert.encoder.layer.3.attention.output.LayerNorm.weight', 'bert.encoder.layer.3.attention.output.LayerNorm.bias', 'bert.encoder.layer.3.output.LayerNorm.weight', 'bert.encoder.layer.3.output.LayerNorm.bias', 'bert.encoder.layer.4.attention.output.La


BERT Classification Report:
                   precision    recall  f1-score   support

 Drug and Alcohol       0.80      0.60      0.69        40
       Early Life       0.62      0.70      0.66        40
      Personality       0.48      0.68      0.56        40
Trauma and Stress       0.59      0.42      0.49        40

         accuracy                           0.60       160
        macro avg       0.62      0.60      0.60       160
     weighted avg       0.62      0.60      0.60       160


BERT → Accuracy: 0.6 | F1: 0.5999


In [11]:
# ── RoBERTa fine-tuning and evaluation ────────────
print("Loading RoBERTa...")
roberta_tokenizer = RobertaTokenizer.from_pretrained('roberta-base')
roberta_model     = RobertaForSequenceClassification.from_pretrained(
    'roberta-base', num_labels=4,
    id2label=id2label, label2id=label2id,
    ignore_mismatched_sizes=True
)

roberta_train = MentalHealthDataset(
    train_df['bert_text'], train_df['label_id'], roberta_tokenizer
)
roberta_test  = MentalHealthDataset(
    test_df['bert_text'], test_df['label_id'], roberta_tokenizer
)

roberta_trainer = Trainer(
    model           = roberta_model,
    args            = get_training_args('/content/roberta_results'),
    train_dataset   = roberta_train,
    eval_dataset    = roberta_test,
    compute_metrics = compute_metrics
)

print("Training RoBERTa — ~20 minutes on GPU\n")
roberta_trainer.train()

# Evaluate
roberta_preds       = roberta_trainer.predict(roberta_test)
roberta_pred_labels = roberta_preds.predictions.argmax(-1)
roberta_true_labels = roberta_preds.label_ids

print("\nRoBERTa Classification Report:")
print(classification_report(
    roberta_true_labels, roberta_pred_labels,
    target_names=list(label2id.keys())
))

roberta_report   = classification_report(roberta_true_labels, roberta_pred_labels, output_dict=True)
roberta_results  = {
    'accuracy' : round(accuracy_score(roberta_true_labels, roberta_pred_labels), 4),
    'f1_macro' : round(roberta_report['macro avg']['f1-score'], 4),
    'precision': round(roberta_report['macro avg']['precision'], 4),
    'recall'   : round(roberta_report['macro avg']['recall'], 4)
}
print(f"\nRoBERTa → Accuracy: {roberta_results['accuracy']} | F1: {roberta_results['f1_macro']}")

# Free GPU memory
del roberta_model, roberta_trainer
gc.collect()
torch.cuda.empty_cache()

Loading RoBERTa...


tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

...

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

There were missing keys in the checkpoint model loaded: ['roberta.embeddings.LayerNorm.weight', 'roberta.embeddings.LayerNorm.bias', 'roberta.encoder.layer.0.attention.output.LayerNorm.weight', 'roberta.encoder.layer.0.attention.output.LayerNorm.bias', 'roberta.encoder.layer.0.output.LayerNorm.weight', 'roberta.encoder.layer.0.output.LayerNorm.bias', 'roberta.encoder.layer.1.attention.output.LayerNorm.weight', 'roberta.encoder.layer.1.attention.output.LayerNorm.bias', 'roberta.encoder.layer.1.output.LayerNorm.weight', 'roberta.encoder.layer.1.output.LayerNorm.bias', 'roberta.encoder.layer.2.attention.output.LayerNorm.weight', 'roberta.encoder.layer.2.attention.output.LayerNorm.bias', 'roberta.encoder.layer.2.output.LayerNorm.weight', 'roberta.encoder.layer.2.output.LayerNorm.bias', 'roberta.encoder.layer.3.attention.output.LayerNorm.weight', 'roberta.encoder.layer.3.attention.output.LayerNorm.bias', 'roberta.encoder.layer.3.output.LayerNorm.weight', 'roberta.encoder.layer.3.output.Laye


RoBERTa Classification Report:
                   precision    recall  f1-score   support

 Drug and Alcohol       0.81      0.95      0.87        40
       Early Life       0.73      0.82      0.78        40
      Personality       0.76      0.65      0.70        40
Trauma and Stress       0.68      0.57      0.62        40

         accuracy                           0.75       160
        macro avg       0.75      0.75      0.74       160
     weighted avg       0.75      0.75      0.74       160


RoBERTa → Accuracy: 0.75 | F1: 0.7436


In [21]:
# ── MentalBERT fine-tuning and evaluation ─────────


print("Loading MentalBERT...")

mental_tokenizer = BertTokenizer.from_pretrained(
    'mental/mental-bert-base-uncased'
)
mental_model     = BertForSequenceClassification.from_pretrained(
    'mental/mental-bert-base-uncased', num_labels=4,
    id2label=id2label, label2id=label2id,
    ignore_mismatched_sizes=True
)

mental_train = MentalHealthDataset(
    train_df['bert_text'], train_df['label_id'], mental_tokenizer
)
mental_test  = MentalHealthDataset(
    test_df['bert_text'], test_df['label_id'], mental_tokenizer
)

mental_trainer = Trainer(
    model           = mental_model,
    args            = get_training_args('/content/mental_results'),
    train_dataset   = mental_train,
    eval_dataset    = mental_test,
    compute_metrics = compute_metrics
)

print("Training MentalBERT — ~20 minutes on GPU\n")
mental_trainer.train()

# Evaluate
mental_preds       = mental_trainer.predict(mental_test)
mental_pred_labels = mental_preds.predictions.argmax(-1)
mental_true_labels = mental_preds.label_ids

print("\nMentalBERT Classification Report:")
print(classification_report(
    mental_true_labels, mental_pred_labels,
    target_names=list(label2id.keys())
))

mental_report  = classification_report(mental_true_labels, mental_pred_labels, output_dict=True)
mental_results = {
    'accuracy' : round(accuracy_score(mental_true_labels, mental_pred_labels), 4),
    'f1_macro' : round(mental_report['macro avg']['f1-score'], 4),
    'precision': round(mental_report['macro avg']['precision'], 4),
    'recall'   : round(mental_report['macro avg']['recall'], 4)
}
print(f"\nMentalBERT → Accuracy: {mental_results['accuracy']} | F1: {mental_results['f1_macro']}")

# Free GPU memory
del mental_model, mental_trainer
gc.collect()
torch.cuda.empty_cache()

Loading MentalBERT...


tokenizer_config.json:   0%|          | 0.00/321 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

...

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

There were missing keys in the checkpoint model loaded: ['bert.embeddings.LayerNorm.weight', 'bert.embeddings.LayerNorm.bias', 'bert.encoder.layer.0.attention.output.LayerNorm.weight', 'bert.encoder.layer.0.attention.output.LayerNorm.bias', 'bert.encoder.layer.0.output.LayerNorm.weight', 'bert.encoder.layer.0.output.LayerNorm.bias', 'bert.encoder.layer.1.attention.output.LayerNorm.weight', 'bert.encoder.layer.1.attention.output.LayerNorm.bias', 'bert.encoder.layer.1.output.LayerNorm.weight', 'bert.encoder.layer.1.output.LayerNorm.bias', 'bert.encoder.layer.2.attention.output.LayerNorm.weight', 'bert.encoder.layer.2.attention.output.LayerNorm.bias', 'bert.encoder.layer.2.output.LayerNorm.weight', 'bert.encoder.layer.2.output.LayerNorm.bias', 'bert.encoder.layer.3.attention.output.LayerNorm.weight', 'bert.encoder.layer.3.attention.output.LayerNorm.bias', 'bert.encoder.layer.3.output.LayerNorm.weight', 'bert.encoder.layer.3.output.LayerNorm.bias', 'bert.encoder.layer.4.attention.output.La


MentalBERT Classification Report:
                   precision    recall  f1-score   support

 Drug and Alcohol       0.80      0.93      0.86        40
       Early Life       0.75      0.75      0.75        40
      Personality       0.63      0.60      0.62        40
Trauma and Stress       0.64      0.57      0.61        40

         accuracy                           0.71       160
        macro avg       0.71      0.71      0.71       160
     weighted avg       0.71      0.71      0.71       160


MentalBERT → Accuracy: 0.7125 | F1: 0.7078


In [22]:
# ── Complete model comparison ──────────────────────
print("=== COMPLETE MODEL COMPARISON ===\n")
print(f"{'Model':<14} {'Accuracy':>10} {'F1 Macro':>10} {'Precision':>10} {'Recall':>10}")
print("-" * 58)
print(f"{'VADER':<14} {'N/A':>10} {'N/A':>10} {'N/A':>10} {'N/A':>10}")
print(f"{'BERT':<14} {bert_results['accuracy']:>10} {bert_results['f1_macro']:>10} {bert_results['precision']:>10} {bert_results['recall']:>10}")
print(f"{'RoBERTa':<14} {roberta_results['accuracy']:>10} {roberta_results['f1_macro']:>10} {roberta_results['precision']:>10} {roberta_results['recall']:>10}")
print(f"{'MentalBERT':<14} {mental_results['accuracy']:>10} {mental_results['f1_macro']:>10} {mental_results['precision']:>10} {mental_results['recall']:>10}")

# Best model
best_f1    = max(bert_results['f1_macro'], roberta_results['f1_macro'], mental_results['f1_macro'])
best_model = ['BERT','RoBERTa','MentalBERT'][
    [bert_results['f1_macro'], roberta_results['f1_macro'], mental_results['f1_macro']].index(best_f1)
]
print(f"\nBest model: {best_model} (F1: {best_f1})")

=== COMPLETE MODEL COMPARISON ===

Model            Accuracy   F1 Macro  Precision     Recall
----------------------------------------------------------
VADER                 N/A        N/A        N/A        N/A
BERT                  0.6     0.5999     0.6226        0.6
RoBERTa              0.75     0.7436     0.7458       0.75
MentalBERT         0.7125     0.7078     0.7062     0.7125

Best model: RoBERTa (F1: 0.7436)


In [24]:
# ── Save all predictions and metrics ──────────────

# BERT predictions
bert_pred_df = test_df.copy()
bert_pred_df['bert_predicted'] = [id2label[p] for p in bert_pred_labels]
bert_pred_df['bert_correct']   = bert_pred_df['Label'] == bert_pred_df['bert_predicted']
bert_pred_df.to_csv(SAVE_PATH + "bert_predictions.csv", index=False)
print("✓ bert_predictions.csv saved")

# RoBERTa predictions
roberta_pred_df = test_df.copy()
roberta_pred_df['roberta_predicted'] = [id2label[p] for p in roberta_pred_labels]
roberta_pred_df['roberta_correct']   = roberta_pred_df['Label'] == roberta_pred_df['roberta_predicted']
roberta_pred_df.to_csv(SAVE_PATH + "roberta_predictions.csv", index=False)
print("✓ roberta_predictions.csv saved")

# MentalBERT predictions
mental_pred_df = test_df.copy()
mental_pred_df['mental_bert_predicted'] = [id2label[p] for p in mental_pred_labels]
mental_pred_df['mental_bert_correct']   = mental_pred_df['Label'] == mental_pred_df['mental_bert_predicted']
mental_pred_df.to_csv(SAVE_PATH + "mental_bert_predictions.csv", index=False)
print("✓ mental_bert_predictions.csv saved")

# Metrics summary
metrics_summary = {
    'BERT'      : bert_results,
    'RoBERTa'   : roberta_results,
    'MentalBERT': mental_results
}
with open(SAVE_PATH + "model_metrics.json", 'w') as f:
    json.dump(metrics_summary, f, indent=2)
print("✓ model_metrics.json saved")

✓ bert_predictions.csv saved
✓ roberta_predictions.csv saved
✓ mental_bert_predictions.csv saved
✓ model_metrics.json saved
